# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata and dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Cite as: {metadata.cite_as}\n")
print(f"Published on: {metadata.date_published}\n")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their '@id'
record_sets = list(dataset.metadata.record_sets)
print(f"Available record sets: {len(record_sets)}\n")
for idx, record_set in enumerate(record_sets):
    print(f"{idx+1}. Record set ID: {record_set.id} - Name: {record_set.name}")

# For demonstration, show fields for the first record set
if record_sets:
    main_rs = record_sets[0]
    print(f"\nFields in record set '{main_rs.name}' (ID: {main_rs.id}):")
    for field in main_rs.fields:
        print(f"  - Field: {field.name} (ID: {field.id}, DataType: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose which record sets to extract
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# We will extract records for each record set
for rset_id in record_set_ids:
    recs = list(dataset.records(record_set=rset_id))
    # Defensive: Try to create a DataFrame if records exist
    if recs:
        dataframes[rset_id] = pd.DataFrame(recs)
        print(f"Loaded DataFrame for record set {rset_id}: {dataframes[rset_id].shape}")
    else:
        print(f"No records found for record set {rset_id}.")

# For demonstration, select the main record set for further work
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nColumns in main DataFrame ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For the EDA, pick a record set and select numeric fields
import numpy as np

# We'll try to find likely numeric fields by name
eda_df = None
if main_record_set_id and main_record_set_id in dataframes:
    eda_df = dataframes[main_record_set_id].copy()
    numeric_candidates = [c for c in eda_df.columns if any(x in c.lower() for x in ['abundance', 'coverage', 'count', 'mw', 'pi', 'intensity'])]
    if not numeric_candidates:
        print("No obvious numeric field found in record set for EDA.")
    else:
        print(f"Numeric candidates detected: {numeric_candidates}")
        # Try to select the first one
        numeric_field = numeric_candidates[0] 

        # Force convert to numeric, coerce errors
        eda_df[numeric_field] = pd.to_numeric(eda_df[numeric_field], errors='coerce')
        threshold = eda_df[numeric_field].quantile(0.75) # Use Q3 as a demonstration threshold

        filtered_df = eda_df[eda_df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (Q3): {len(filtered_df)} records\n")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field
        group_candidates = [c for c in eda_df.columns if any(x in c.lower() for x in ['sample', 'type', 'group', 'accession'])]
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No appropriate group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualization (histogram and boxplot) for the detected numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if eda_df is not None and numeric_candidates:
    plt.figure(figsize=(14,5))
    plt.subplot(1,2,1)
    sns.histplot(eda_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)

    plt.subplot(1,2,2)
    sns.boxplot(data=eda_df, x=numeric_field)
    plt.title(f"Boxplot of '{numeric_field}'")
    plt.show()

    # If group_field detected, visualize group means
    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No numeric field found to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we loaded and explored the FAIR\textsuperscript{2} mass spectrometry dataset of extracellular vesicles using the `mlcroissant` library. We programmatically discovered the available record sets and fields, loaded tabular data, performed basic EDA including normalization and grouping, and visualized the distributions of key numeric attributes. For more advanced or domain-specific analysis, consult the dataset metadata and documentation for precise field definitions and recommended workflows.*